# Colab GPU → API de LLM na sua tailnet

Este notebook:
1. Sobe o **Ollama** na GPU do Colab;
2. Coloca a VM do Colab na sua **tailnet** (Tailscale em modo userspace, já que o Colab não tem `/dev/net/tun`);
3. Expõe a API em `http://<HOSTNAME>:11434` — nativa do Ollama e compatível com OpenAI em `/v1`.

**Antes de rodar:**
- Ambiente de execução → Alterar tipo → **GPU** (T4, L4 ou A100).
- No admin do Tailscale gere uma *auth key* **reutilizável + efêmera** (de preferência com tag, ex. `tag:colab`).
- No Colab, ícone 🔑 *Secrets* → crie `TS_AUTHKEY` com a chave e habilite o acesso do notebook.

Depois é só **Executar tudo** e deixar a última célula rodando.

In [ ]:
#@title Configuração
TS_HOSTNAME = "colab-gpu"  #@param {type:"string"}
MODELS = "qwen2.5:7b"  #@param {type:"string"}
#@markdown Separe vários modelos por vírgula. Ex.: `qwen2.5:7b,llama3.1:8b,nomic-embed-text`
USE_DRIVE_CACHE = False  #@param {type:"boolean"}
#@markdown Guarda os modelos no Google Drive para não baixar de novo a cada sessão (precisa de espaço no Drive).
OLLAMA_PORT = 11434  #@param {type:"integer"}
OLLAMA_CONTEXT_LENGTH = 8192  #@param {type:"integer"}

MODEL_LIST = [m.strip() for m in MODELS.split(",") if m.strip()]

In [ ]:
#@title Verifica GPU
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
#@title (Opcional) Monta o Drive para cache de modelos
import os
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    MODELS_DIR = "/content/drive/MyDrive/ollama-models"
else:
    MODELS_DIR = "/content/ollama-models"
os.makedirs(MODELS_DIR, exist_ok=True)
print("Modelos em:", MODELS_DIR)

In [ ]:
#@title Instala e conecta o Tailscale
import subprocess, time, os
from google.colab import userdata

TS_AUTHKEY = userdata.get("TS_AUTHKEY")

if subprocess.run(["which", "tailscale"], capture_output=True).returncode != 0:
    !curl -fsSL https://tailscale.com/install.sh | sh > /tmp/ts-install.log 2>&1

os.makedirs("/run/tailscale", exist_ok=True)
if subprocess.run(["pgrep", "-x", "tailscaled"], capture_output=True).returncode != 0:
    tailscaled = subprocess.Popen(
        ["tailscaled", "--tun=userspace-networking", "--state=mem:",
         "--socket=/run/tailscale/tailscaled.sock"],
        stdout=open("/tmp/tailscaled.log", "a"), stderr=subprocess.STDOUT,
    )
    time.sleep(3)

subprocess.run(
    ["tailscale", "up", f"--authkey={TS_AUTHKEY}", f"--hostname={TS_HOSTNAME}",
     "--accept-dns=false", "--reset"],
    check=True,
)
del TS_AUTHKEY
!tailscale ip -4
!tailscale status | head -n 5

In [ ]:
#@title Instala e inicia o Ollama
import subprocess, time, os, urllib.request

!apt-get -qq install -y zstd pciutils lshw > /dev/null 2>&1
if subprocess.run(["which", "ollama"], capture_output=True).returncode != 0:
    !curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama-install.log 2>&1

OLLAMA_ENV = {
    **os.environ,
    # 0.0.0.0: o netstack do tailscaled entrega as conexões da tailnet no loopback,
    # mas escutar em todas as interfaces evita surpresas. A VM do Colab não é exposta publicamente.
    "OLLAMA_HOST": f"0.0.0.0:{OLLAMA_PORT}",
    "OLLAMA_MODELS": MODELS_DIR,
    "OLLAMA_ORIGINS": "*",
    "OLLAMA_KEEP_ALIVE": "-1",
    "OLLAMA_CONTEXT_LENGTH": str(OLLAMA_CONTEXT_LENGTH),
    "OLLAMA_FLASH_ATTENTION": "1",
}

def start_ollama():
    return subprocess.Popen(["ollama", "serve"], env=OLLAMA_ENV,
                            stdout=open("/tmp/ollama.log", "a"), stderr=subprocess.STDOUT)

def ollama_up():
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{OLLAMA_PORT}/api/version", timeout=3)
        return True
    except Exception:
        return False

if not ollama_up():
    ollama_proc = start_ollama()
    for _ in range(30):
        if ollama_up():
            break
        time.sleep(1)
print("Ollama no ar:", ollama_up())
!grep -iE "inference compute|cuda" /tmp/ollama.log | tail -n 3

In [ ]:
#@title Baixa os modelos
for m in MODEL_LIST:
    print(f"==> {m}")
    subprocess.run(["ollama", "pull", m], env=OLLAMA_ENV, check=True)
!OLLAMA_HOST=127.0.0.1:{OLLAMA_PORT} ollama list

In [ ]:
#@title Teste local + endereço para usar na tailnet
import json
ts_ip = subprocess.run(["tailscale", "ip", "-4"], capture_output=True, text=True).stdout.strip()
body = json.dumps({"model": MODEL_LIST[0],
                   "messages": [{"role": "user", "content": "Diga oi em uma frase."}]}).encode()
req = urllib.request.Request(f"http://127.0.0.1:{OLLAMA_PORT}/v1/chat/completions", data=body,
                             headers={"Content-Type": "application/json"})
print(json.load(urllib.request.urlopen(req, timeout=300))["choices"][0]["message"]["content"])

print(f"""
API disponível na tailnet:
  Ollama nativo : http://{TS_HOSTNAME}:{OLLAMA_PORT}/api   (ou http://{ts_ip}:{OLLAMA_PORT})
  OpenAI-compat : http://{TS_HOSTNAME}:{OLLAMA_PORT}/v1
""")

## Mantenha a célula abaixo rodando
Ela monitora o Ollama e o Tailscale, reinicia o Ollama se cair e imprime o status a cada minuto.
Parar a célula **não** derruba a API; fechar/desconectar a sessão do Colab derruba (e a chave efêmera remove o nó da tailnet).

In [ ]:
#@title Watchdog
import datetime
while True:
    ok_ts = subprocess.run(["tailscale", "status", "--peers=false"], capture_output=True).returncode == 0
    if not ollama_up():
        print("Ollama caiu, reiniciando...")
        ollama_proc = start_ollama()
        time.sleep(5)
    loaded = subprocess.run(["ollama", "ps"], env=OLLAMA_ENV, capture_output=True, text=True).stdout.strip().splitlines()[1:]
    print(f"[{datetime.datetime.now():%H:%M:%S}] tailscale={'ok' if ok_ts else 'FALHA'} "
          f"ollama={'ok' if ollama_up() else 'FALHA'} carregados={[l.split()[0] for l in loaded]}", flush=True)
    time.sleep(60)